In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [2]:
df = pd.read_csv("data/clean_soccer_dataset.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

df.head()

,date,year,decade,home_team,away_team,home_team_code,away_team_code,home_score,away_score,match_result,matchday_name,round,type,match_attendance,stadium_country_code,stadium_capacity
0,1958-09-28,1958,1950,USSR,Hungary,URS,HUN,3.0,1.0,H,1,ROUND_OF_16,FIRST_LEG,100572.0,RUS,81015.0
1,1958-10-01,1958,1950,France,Greece,FRA,GRE,7.0,1.0,H,1,ROUND_OF_16,FIRST_LEG,37590.0,FRA,47926.0
2,1958-11-02,1958,1950,Romania,Türki̇ye,ROU,TUR,3.0,0.0,H,1,ROUND_OF_16,FIRST_LEG,67200.0,ROU,0.0
3,1958-12-03,1958,1950,Greece,France,GRE,FRA,1.0,1.0,D,2,ROUND_OF_16,SECOND_LEG,18833.0,GRE,68224.0
4,1959-04-05,1959,1950,Republic of Ireland,Czechoslovakia,IRL,TCH,2.0,0.0,H,1,PRELIMINARY,FIRST_LEG,37500.0,IRL,2740.0


In [3]:
# Win indicators
df["home_win"] = (df["match_result"] == "H").astype(int)
df["away_win"] = (df["match_result"] == "A").astype(int)

# Goals conceded
df["home_conceded"] = df["away_score"]
df["away_conceded"] = df["home_score"]

In [4]:
df["home_win_rate_5"] = (
    df.groupby("home_team")["home_win"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

df["home_goals_avg_5"] = (
    df.groupby("home_team")["home_score"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

df["home_conceded_avg_5"] = (
    df.groupby("home_team")["home_conceded"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

In [5]:
df["away_win_rate_5"] = (
    df.groupby("away_team")["away_win"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

df["away_goals_avg_5"] = (
    df.groupby("away_team")["away_score"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

df["away_conceded_avg_5"] = (
    df.groupby("away_team")["away_conceded"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

In [6]:
df[[
    "home_win_rate_5",
    "away_win_rate_5"
]].head(10)

,home_win_rate_5,away_win_rate_5
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN
6,NaN,NaN
7,NaN,NaN
8,NaN,NaN
9,NaN,NaN


In [7]:
df[["home_team", "home_win_rate_5"]].dropna().head(10)

,home_team,home_win_rate_5
84,France,0.8
91,Republic of Ireland,0.6
98,Spain,0.6
110,Hungary,0.6
114,Republic of Ireland,0.6
115,Denmark,0.6
117,Spain,0.6
119,Denmark,0.6
124,Romania,0.8
126,USSR,1.0


In [8]:
df_model = df.dropna().copy()

print("Shape after rolling:", df_model.shape)

Shape after rolling: (2372, 26)


In [11]:
features = [
    "home_win_rate_5",
    "away_win_rate_5",
    "home_goals_avg_5",
    "away_goals_avg_5",
    "home_conceded_avg_5",
    "away_conceded_avg_5"
]

X = df_model[features]
y = df_model["match_result"]

In [12]:
df_model = df_model.sort_values("date")

split_index = int(0.8 * len(df_model))

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [13]:
from sklearn.preprocessing import LabelEncoder

le_target = LabelEncoder()
y_train_enc = le_target.fit_transform(y_train)
y_test_enc = le_target.transform(y_test)

In [14]:
from xgboost import XGBClassifier

model_xgb_new = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42
)

model_xgb_new.fit(X_train, y_train_enc)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [15]:
y_pred_xgb_new = model_xgb_new.predict(X_test)

# Convert back to original labels (H, D, A)
y_pred_xgb_new_labels = le_target.inverse_transform(y_pred_xgb_new)

In [16]:
from sklearn.metrics import accuracy_score, classification_report

print("New XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb_new_labels))

print(classification_report(y_test, y_pred_xgb_new_labels))

New XGBoost Accuracy: 0.6484210526315789
              precision    recall  f1-score   support

           A       0.60      0.78      0.68       162
           D       0.24      0.07      0.11        82
           H       0.74      0.76      0.75       231

    accuracy                           0.65       475
   macro avg       0.52      0.54      0.51       475
weighted avg       0.60      0.65      0.61       475



Some other ways to make the result better

In [17]:
# relative strength features
df_model["win_rate_diff_5"] = df_model["home_win_rate_5"] - df_model["away_win_rate_5"]
df_model["goals_avg_diff_5"] = df_model["home_goals_avg_5"] - df_model["away_goals_avg_5"]
df_model["conceded_avg_diff_5"] = df_model["home_conceded_avg_5"] - df_model["away_conceded_avg_5"]

# team balance features
df_model["goal_balance_home_5"] = df_model["home_goals_avg_5"] - df_model["home_conceded_avg_5"]
df_model["goal_balance_away_5"] = df_model["away_goals_avg_5"] - df_model["away_conceded_avg_5"]

# match context
df_model["attendance_ratio"] = df_model["match_attendance"] / df_model["stadium_capacity"]
df_model["attendance_ratio"] = df_model["attendance_ratio"].replace([np.inf, -np.inf], np.nan)

In [18]:
new_feature_cols = [
    "win_rate_diff_5",
    "goals_avg_diff_5",
    "conceded_avg_diff_5",
    "goal_balance_home_5",
    "goal_balance_away_5",
    "attendance_ratio"
]

df_model[new_feature_cols].isnull().sum()

win_rate_diff_5         0
goals_avg_diff_5        0
conceded_avg_diff_5     0
goal_balance_home_5     0
goal_balance_away_5     0
attendance_ratio       55
dtype: int64

In [19]:
df_model["attendance_ratio"] = df_model["attendance_ratio"].fillna(df_model["attendance_ratio"].median())

In [20]:
features_improved = [
    "home_win_rate_5",
    "away_win_rate_5",
    "home_goals_avg_5",
    "away_goals_avg_5",
    "home_conceded_avg_5",
    "away_conceded_avg_5",
    "win_rate_diff_5",
    "goals_avg_diff_5",
    "conceded_avg_diff_5",
    "goal_balance_home_5",
    "goal_balance_away_5",
    "attendance_ratio",
    "year",
    "decade"
]

X = df_model[features_improved]
y = df_model["match_result"]

In [21]:
df_model = df_model.sort_values("date").reset_index(drop=True)

X = df_model[features_improved]
y = df_model["match_result"]

split_index = int(0.8 * len(df_model))

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [22]:
from sklearn.preprocessing import LabelEncoder

le_target = LabelEncoder()
y_train_enc = le_target.fit_transform(y_train)
y_test_enc = le_target.transform(y_test)

In [23]:
from xgboost import XGBClassifier

model_xgb_improved = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

model_xgb_improved.fit(X_train, y_train_enc)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [24]:
y_pred_xgb_improved = model_xgb_improved.predict(X_test)
y_pred_xgb_improved_labels = le_target.inverse_transform(y_pred_xgb_improved)

from sklearn.metrics import accuracy_score, classification_report

print("Improved XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb_improved_labels))
print(classification_report(y_test, y_pred_xgb_improved_labels))

Improved XGBoost Accuracy: 0.6694736842105263
              precision    recall  f1-score   support

           A       0.61      0.81      0.70       162
           D       0.40      0.05      0.09        82
           H       0.73      0.79      0.76       231

    accuracy                           0.67       475
   macro avg       0.58      0.55      0.51       475
weighted avg       0.63      0.67      0.62       475

